# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Load the anonymized starter file that ships with the repo.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AsserGharib1/flyrank-internshipML"
REPO_DIR = "flyrank-internshipML"

def at_repo_root():
    return os.path.isdir("data/raw")

if IN_COLAB:
    if not at_repo_root():
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Walk up to the repo root. Stop when the directory stops changing, which is
    # what happens at a drive root on Windows and at / on Linux.
    previous = None
    while not at_repo_root() and os.getcwd() != previous:
        previous = os.getcwd()
        os.chdir("..")

assert at_repo_root(), "Could not find data/raw. Open this from inside the repo."

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

pd.set_option("display.width", 140)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df), " Columns:", df.shape[1], " Clients:", df.client_id.nunique())

Rows: 30000  Columns: 44  Clients: 32


## 1. My lane (or freestyle) and why

My provisional choice is **Lane 2 — Refresh / Content Opportunity Scoring**. I want to rank pages for human review using only information available at the decision time, then attach a reason and a suggested review action. The starter data shows that a broad decline flag leaves too many candidates, so prioritisation is the useful problem to test.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with enough observed demand, which pages should a reviewer inspect first for a content opportunity, and can a learned ranking beat a transparent rule on the same held-out data?

**Unit of analysis:** one eligible content page at a weekly decision point.

**Decision and action:** The output is a weekly review queue. A content or SEO reviewer decides whether a page needs refresh, expansion, protection, pruning, monitoring, or no action.

**Cost of a wrong call:** A false positive uses limited review capacity on a page that did not need attention. A false negative leaves a potentially useful review outside the queue. Because the reviewer only opens the top of the list, the ranking at a fixed review capacity matters more than whole-dataset accuracy.

## 3. Quick look at the data (2-3 real numbers)

I use a provisional floor of 100 impressions in the earlier 30-day window so the first comparison is made on pages with some observed demand. The floor is a modelling choice, not a claim that lower-volume pages are useless.

In [2]:
MIN_PRIOR_IMPRESSIONS = 100

eligible = df[df["impressions_prev_30d"] >= MIN_PRIOR_IMPRESSIONS].copy()
eligible["change_pct"] = (
    (eligible["impressions_last_30d"] - eligible["impressions_prev_30d"])
    / eligible["impressions_prev_30d"] * 100
)
client_median = eligible.groupby("client_id")["change_pct"].transform("median")
eligible["gap_vs_client"] = eligible["change_pct"] - client_median

print(f"1. Pages above the provisional demand floor: {len(eligible):,} across {eligible.client_id.nunique()} clients")
print(f"2. Share falling more than 20%: {(eligible.change_pct < -20).mean():.1%}")
print(f"3. Share falling at least 20 points behind their own client median: {(eligible.gap_vs_client <= -20).mean():.1%}")

1. Pages above the provisional demand floor: 18,010 across 30 clients
2. Share falling more than 20%: 61.6%
3. Share falling at least 20 points behind their own client median: 25.1%


## 4. Careful words: what I can and can't claim

**I can say:** these patterns were observed in an anonymized starter sample, and later I can compare a ranking with a rule baseline under the same validation design.

**I cannot say:** that a refresh caused recovery, that I reverse engineered Google ranking, or that a high score guarantees a page should be changed. The final queue is decision support for a human reviewer.

## Self-check

- [x] One lane and one research question are stated clearly
- [x] The decision, action, and cost of a wrong call are named
- [x] Three starter-data numbers support the lane choice
- [x] Claims stay observational and public-safe
- [ ] Final corrected notebook committed and repo URL submitted on the ML-02 card